In [1]:
import boto3
import os
import tarfile

from dotenv import load_dotenv
from sagemaker.model import Model

load_dotenv(override=True)

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\alejo\AppData\Local\sagemaker\sagemaker\config.yaml


True

In [6]:
with tarfile.open("rf_model.tar.gz", "w:gz") as tar:
    tar.add("../local-server/model/rf-final-model.pkl", arcname="rf_model.pkl")

In [2]:
# Configuración del bucket de s3
s3 = boto3.client('s3')
bucket_name = os.getenv("BUCKET_NAME")
model_path = "model/rf_model.tar.gz"

In [7]:
# Carga del modelo en el bucket
s3.upload_file("rf_model.tar.gz", bucket_name, model_path)

In [ ]:
# Luego de crear el bucket, creé una imagen de docker con el modelo
# Probé de manera local el modelo
# Luego creé un repositorio en ECR para guardar la imagen

In [9]:
image_uri = os.getenv("IMAGE_URI")
role = os.getenv("ROLE_ARN")
model_data = f"s3://{bucket_name}/{model_path}"

model = Model(image_uri=image_uri, model_data=model_data, role=role)

In [10]:
# Desplegar el endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="rf-model-endpoint"
)

----!

In [13]:
# Crear cliente de SageMaker Runtime
runtime = boto3.client('sagemaker-runtime')

payload = '{"rooms": 5,"bedrooms": 2,"bathrooms": 1,"surface_total": 65,"surface_covered": 50,"l2": "G.B.A Zona Norte","l3": "Berazategui","property_type": "Casa"}'

# Enviar la solicitud al endpoint
response = runtime.invoke_endpoint(
    EndpointName="rf-model-endpoint",
    ContentType="application/json",
    Body=payload
)

# Imprimir la predicción
print("Predicción:", response['Body'].read().decode("utf-8"))

Predicción: {"Price_in_USD":103666.67}
